# 🚀 ISIRI 2.0 — ByT5-Small Bidirectional Translation Fine-Tuning
### **Romanised Tulu ↔ English Neural Machine Translation**

This notebook fine-tunes **`google/byt5-small`** for bidirectional translation between Romanised Tulu and English.

#### **Why ByT5?**
- Operates directly on raw UTF-8 byte sequences (no fixed subword vocabulary).
- Completely eliminates `<UNK>` out-of-vocabulary errors caused by spelling variations in Romanised Tulu (`malpule`, `malpu`, `malpulet`, `mlpule`).
- Enables bidirectional multi-task translation via prefix prompts:
  - `translate Tulu to English: <tulu>` $\rightarrow$ `<english>`
  - `translate English to Tulu: <english>` $\rightarrow$ `<tulu>`

## 1. Verify GPU & Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi

# Install required packages
!pip install -q transformers datasets accelerate evaluate sacrebleu sentencepiece torch pandas

## 2. Load & Prepare Bidirectional Dataset
Upload `clean_dataset.csv` from your local `datasets/processed/` folder if running on Google Colab.

In [ ]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict
from google.colab import files

DATASET_FILE = "clean_dataset.csv"

if not os.path.exists(DATASET_FILE):
    print("Please upload clean_dataset.csv:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        DATASET_FILE = fn

df = pd.read_csv(DATASET_FILE)
df = df.dropna(subset=["English", "Tulu"])
df["English"] = df["English"].astype(str).str.strip()
df["Tulu"] = df["Tulu"].astype(str).str.strip()
df = df[(df["English"] != "") & (df["Tulu"] != "")]

print(f"Loaded {len(df)} parallel sentence pairs.")

# Generate bidirectional examples
inputs = []
targets = []
directions = []

for _, row in df.iterrows():
    en_text = row["English"]
    tu_text = row["Tulu"]

    # Tulu -> English
    inputs.append(f"translate Tulu to English: {tu_text}")
    targets.append(en_text)
    directions.append("tulu_to_en")

    # English -> Tulu
    inputs.append(f"translate English to Tulu: {en_text}")
    targets.append(tu_text)
    directions.append("en_to_tulu")

full_dataset = Dataset.from_dict({
    "input_text": inputs,
    "target_text": targets,
    "direction": directions
})

# 80% Train, 10% Validation, 10% Test
split_1 = full_dataset.train_test_split(test_size=0.2, seed=42)
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    "train": split_1["train"],
    "validation": split_2["train"],
    "test": split_2["test"]
})

print(f"Split counts -> Train: {len(dataset_dict['train'])}, Val: {len(dataset_dict['validation'])}, Test: {len(dataset_dict['test'])}")
print("Sample training instance:", dataset_dict["train"][0])

## 3. Load Tokenizer & Tokenize Datasets

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 256

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text", "direction"]
)
print("Tokenization complete!")

## 4. Setup Model, Metrics & Seq2SeqTrainer

In [ ]:
import numpy as np
import torch
import evaluate
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

sacrebleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

# Direct safe byte decoding for ByT5
def decode_byt5_sequences(sequences):
    decoded = []
    for seq in sequences:
        raw_bytes = bytearray([int(t) - 3 for t in seq if 3 <= int(t) <= 258])
        decoded.append(raw_bytes.decode("utf-8", errors="ignore").strip())
    return decoded

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    decoded_preds = decode_byt5_sequences(preds)
    decoded_labels = [[l] for l in decode_byt5_sequences(labels)]
    
    bleu = sacrebleu_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"]
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)["score"]
    
    return {
        "bleu": round(bleu, 2),
        "chrf": round(chrf, 2)
    }

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

OUTPUT_DIR = "./byt5_tulu_english_model"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=15,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=3,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

## 5. Train Model

In [ ]:
print("Starting training on GPU...")
trainer.train()

## 6. Evaluate on Test Set

In [ ]:
test_results = trainer.predict(
    test_dataset=tokenized_datasets["test"],
    metric_key_prefix="test"
)
print(f"\n=== TEST SET PERFORMANCE ===")
print(f"Test SacreBLEU: {test_results.metrics.get('test_bleu')}")
print(f"Test chrF++:   {test_results.metrics.get('test_chrf')}")

## 7. Interactive Translation Playground

In [ ]:
def translate(text, direction="tulu_to_en"):
    if direction == "tulu_to_en":
        prompt = f"translate Tulu to English: {text}"
    else:
        prompt = f"translate English to Tulu: {text}"
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=MAX_LENGTH, truncation=True).to(model.device)
    outputs = model.generate(**inputs, max_length=MAX_LENGTH, num_beams=3)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test sample translations
test_samples = [
    ("youtube open malpule", "tulu_to_en"),
    ("yencha ullar?", "tulu_to_en"),
    ("ini mangalore da weather encha undu", "tulu_to_en"),
    ("yaan illag povond ulle", "tulu_to_en"),
    ("Turn off the bedroom light", "en_to_tulu"),
    ("What is the weather in Delhi tomorrow?", "en_to_tulu"),
]

print("\n=== SAMPLE INFERENCE ===")
for sentence, direction in test_samples:
    result = translate(sentence, direction)
    print(f"[{direction}] {sentence} -> {result}")

## 8. Export Model for ISIRI 2.0

In [ ]:
# Save model & tokenizer
FINAL_EXPORT = "./byt5_tulu_english"
trainer.save_model(FINAL_EXPORT)
tokenizer.save_pretrained(FINAL_EXPORT)

# Zip the model for easy download
!zip -r byt5_tulu_english.zip ./byt5_tulu_english

print("Model exported successfully as 'byt5_tulu_english.zip'!")
print("Download this file and unzip into 'datasets/models/byt5_tulu_english/' on your local machine.")

# Automatically trigger download in Google Colab
try:
    files.download('byt5_tulu_english.zip')
except Exception as e:
    print("Download trigger:", e)